# Course 8: Mathematics and AI
## Session 2: Representations and irreducibles

**Goal:** decompose a representation of a small group into irreducible pieces with characters and projectors, then split a product of two vectors under rotations.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every Python cell without a NOT RUN label was run once on a CPU (November 2026). Cells labelled NOT RUN need a library, a GPU or a Lean installation. Lean code in this course was written carefully but was NOT compiled here: check it in Lean before you teach it.

### 1. A representation of S3
S3 acts on three objects. Sending each permutation to its 3x3 permutation matrix is a representation: `rho(g h) = rho(g) rho(h)`.

In [ ]:
import itertools, numpy as np
perms = list(itertools.permutations(range(3)))
rho = {p: np.eye(3)[list(p)] for p in perms}
compose = lambda p, q: tuple(q[p[i]] for i in range(3))
assert all(np.allclose(rho[compose(p, q)], rho[p] @ rho[q]) for p in perms for q in perms)
print("rho is a homomorphism: checked on all 36 pairs")

### 2. Characters
The character of a representation is the trace of each matrix. S3 has three irreducible representations: trivial, sign, and a 2-dimensional one. Their characters depend only on the type of the permutation (identity, a swap, a 3-cycle).

In [ ]:
def cycle_type(p): return {3: "identity", 1: "swap", 0: "3-cycle"}[sum(p[i] == i for i in range(3))]
def sign(p): return round(np.linalg.det(rho[p]))
chi = {"trivial": lambda p: 1, "sign": sign, "standard": lambda p: {"identity": 2, "swap": 0, "3-cycle": -1}[cycle_type(p)]}
inner = lambda a, b: sum(a(p) * b(p) for p in perms) / 6
print("character table (rows are orthonormal):")
for n1 in chi: print(f"  {n1:9s}", [int(inner(chi[n1], chi[n2])) for n2 in chi])
chi_perm = lambda p: np.trace(rho[p])
mult = {n: inner(chi_perm, c) for n, c in chi.items()}
print("the permutation representation contains:", {k: int(v) for k, v in mult.items()})
assert mult == {"trivial": 1, "sign": 0, "standard": 1}

So the 3-dimensional representation is trivial + standard: a 1-dimensional piece and a 2-dimensional piece. Characters told us this without finding the pieces.

### 3. Find the pieces with projectors
`P = (dim / |G|) * sum over g of chi(g) * rho(g)` projects onto the part of the space that transforms like the irreducible with character chi.

In [ ]:
dims = {"trivial": 1, "sign": 1, "standard": 2}
proj = {n: dims[n] / 6 * sum(chi[n](p) * rho[p] for p in perms) for n in chi}
for n, Pm in proj.items(): print(f"{n:9s} projector has rank {np.linalg.matrix_rank(Pm)}")
print("trivial piece is spanned by", np.round(proj["trivial"][:, 0] / np.linalg.norm(proj["trivial"][:, 0]), 3), "(the all-ones direction)")

# Change basis: first the trivial direction, then a basis of the standard piece. Every rho(g) becomes block-diagonal.
u, s, _ = np.linalg.svd(proj["standard"]); B = np.c_[np.ones(3) / np.sqrt(3), u[:, :2]]
blocks = {p: np.round(B.T @ rho[p] @ B, 3) for p in perms}
print("rho of a swap in the new basis:"); print(blocks[(1, 0, 2)])
assert all(abs(M[0, 1:]).max() < 1e-9 and abs(M[1:, 0]).max() < 1e-9 for M in blocks.values())

### 4. Why a network designer cares: Schur's lemma
A linear layer that commutes with the group can only act as a multiple of the identity on each irreducible piece. So an equivariant layer on this space has exactly 2 free numbers, not 9.

In [ ]:
# Solve W rho(g) = rho(g) W for all g, as a linear system in the 9 entries of W.
rows = [np.kron(np.eye(3), rho[p]) - np.kron(rho[p].T, np.eye(3)) for p in perms]
_, sv, Vt = np.linalg.svd(np.vstack(rows)); null = Vt[np.sum(sv > 1e-9):]
print("dimension of the space of equivariant linear maps:", len(null))
assert len(null) == 2
W = null[0].reshape(3, 3, order="F"); print("one solution (rounded): equal diagonal, equal off-diagonal"); print(np.round(W / abs(W).max(), 3))

That is the DeepSets layer: `W = a * I + b * (all ones)`. Representation theory derived the architecture.

### 5. The regular representation of C4 and the Fourier transform
C4 acting on functions on its own four elements is a cyclic shift. Its irreducibles are 1-dimensional, and the change of basis that splits it is the discrete Fourier transform.

In [ ]:
shift = np.roll(np.eye(4), 1, axis=0)
Fm = np.fft.fft(np.eye(4)) / 2                                  # unitary DFT matrix
D = Fm @ shift @ Fm.conj().T
print("shift in the Fourier basis is diagonal:", np.allclose(D, np.diag(np.diag(D))), "| diagonal:", np.round(np.diag(D), 3))
assert np.allclose(D, np.diag(np.diag(D)))

### 6. Rotations in 3-D: splitting the product of two vectors
Take two vectors `u` and `v`. Their 9 products `u_i v_j` transform under a rotation R by `R ⊗ R`. That 9-dimensional representation splits into pieces of dimension 1, 3 and 5:
- the dot product (does not change),
- the cross product (rotates like a vector),
- the symmetric traceless part (5 numbers).

In e3nn's notation this is `1o ⊗ 1o = 0e + 1e + 2e`.

In [ ]:
rng = np.random.default_rng(0)
Q, _ = np.linalg.qr(rng.normal(size=(3, 3))); Rm = Q * np.sign(np.linalg.det(Q))       # a random rotation
u, v = rng.normal(size=3), rng.normal(size=3)
def parts(u, v):
    M = np.outer(u, v); scalar = np.trace(M); anti = (M - M.T) / 2; sym = (M + M.T) / 2 - scalar / 3 * np.eye(3)
    return scalar, anti, sym
s1, a1, y1 = parts(u, v); s2, a2, y2 = parts(Rm @ u, Rm @ v)
print("scalar part unchanged       :", np.isclose(s1, s2))
print("antisymmetric part rotates  :", np.allclose(a2, Rm @ a1 @ Rm.T), "| it holds the cross product:", np.allclose([a1[2, 1], a1[0, 2], a1[1, 0]], -np.cross(u, v) / 2))
print("symmetric traceless rotates :", np.allclose(y2, Rm @ y1 @ Rm.T))
print("dimensions: 1 + 3 + 5 =", 1 + 3 + 5)
assert np.isclose(s1, s2) and np.allclose(a2, Rm @ a1 @ Rm.T) and np.allclose(y2, Rm @ y1 @ Rm.T)

### 7. The same with e3nn (NOT RUN here)

In [ ]:
# NOT RUN: needs `pip install e3nn`.
import torch
from e3nn import o3
tp = o3.FullTensorProduct("1o", "1o"); print(tp)                         # expect the output irreps 1x0e+1x1e+1x2e
irreps_in, irreps_out = o3.Irreps("8x0e + 4x1o"), o3.Irreps("8x0e + 4x1o + 2x2e")
layer = o3.FullyConnectedTensorProduct(irreps_in, "1x0e + 1x1o + 1x2e", irreps_out)     # features, spherical harmonics of a direction, output
x = irreps_in.randn(5, -1); r = torch.randn(5, 3); sh = o3.spherical_harmonics([0, 1, 2], r, normalize=True)
rot = o3.rand_matrix()
out_then_rotate = layer(x, sh) @ irreps_out.D_from_matrix(rot).T
rotate_then_out = layer(x @ irreps_in.D_from_matrix(rot).T, o3.spherical_harmonics([0, 1, 2], r @ rot.T, normalize=True))
print("equivariance error:", (out_then_rotate - rotate_then_out).abs().max())

### TODO
1. Decompose the 6-dimensional regular representation of S3 with characters. How many times does each irreducible appear? Compare with its dimension.
2. Repeat section 4 for C4 acting on 4 points. How many free numbers does an equivariant linear layer have, and what kind of matrix is it?
3. Run the e3nn cell. Change the output to include `1e`. What does the parity label mean?